# Week 5 Day 2 — Residual Exploratory Analysis

Three questions for today:

1. **Persistence** — how long do deviations last? (high autocorrelation = tradeable)
2. **On-the-run richness** — do benchmark maturities (2Y/5Y/10Y/30Y) show
   systematic rich or cheap bias?
3. **Distribution shape** — are residuals fat-tailed or skewed in ways that
   affect how we should set z-score thresholds in Day 3?

No new parquet files are produced — this notebook is purely exploratory.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
%matplotlib inline

residuals = pd.read_parquet('../data/processed/bond_residuals.parquet')
residuals['date'] = pd.to_datetime(residuals['date'])

means = pd.read_parquet('../data/processed/residual_means.parquet')
means = means.set_index('maturity')['mean_residual_bps']

MATURITIES  = sorted(residuals['maturity'].unique())
BENCHMARKS  = [2, 5, 10, 30]
OFF_BENCH   = [1, 3, 7, 20]

pivot = residuals.pivot(index='date', columns='maturity', values='residual_bps').sort_index()

print(f'Date range : {residuals["date"].min().date()} -> {residuals["date"].max().date()}')
print(f'Maturities : {MATURITIES}')
print(f'Total rows : {len(residuals):,}')

## 1. Residuals through time

A well-behaved relative-value signal should look like stationary noise around
zero — no long drifts, occasional spikes that eventually revert.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
axes = axes.flatten()

for i, mat in enumerate(MATURITIES):
    ax = axes[i]
    series = pivot[mat].dropna()
    ax.plot(series.index, series.values, lw=0.5, color='steelblue', alpha=0.8)
    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_title(f'{mat}Y  |  std={series.std():.1f} bp  |  ac1={series.autocorr():.3f}')
    ax.set_ylabel('residual (bp)')
    bench_label = '  [benchmark]' if mat in BENCHMARKS else ''
    ax.set_title(f'{mat}Y{bench_label}  |  std={series.std():.1f} bp  |  ac1={series.autocorr():.3f}')

fig.suptitle('Demeaned residuals through time (observed - fitted, bp)', y=1.01)
plt.tight_layout()
plt.savefig('data/week5_day2_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Persistence: autocorrelation and half-life

The **half-life** is how many days it takes a deviation to decay to half its
original size. It's derived from the lag-1 autocorrelation via:

```
half_life = -log(2) / log(ac1)
```

A half-life of 10–30 days is ideal for a weekly-rebalanced strategy: long
enough that the deviation persists after you enter, short enough that it
reverts within a reasonable holding period.

In [ ]:
rows = []
for mat in MATURITIES:
    s = pivot[mat].dropna()
    ac1 = s.autocorr(lag=1)
    half_life = -np.log(2) / np.log(ac1) if ac1 > 0 else np.nan
    rows.append({'maturity': mat, 'ac1': ac1, 'half_life_days': half_life,
                 'is_benchmark': mat in BENCHMARKS})

ac_df = pd.DataFrame(rows).set_index('maturity')
print(ac_df.round(1))

In [ ]:
# Plot autocorrelation up to lag 60 for benchmark vs off-benchmark
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, group, title in zip(axes, [BENCHMARKS, OFF_BENCH],
                             ['Benchmark maturities (2Y/5Y/10Y/30Y)',
                              'Off-benchmark (1Y/3Y/7Y/20Y)']):
    for mat in group:
        s = pivot[mat].dropna()
        lags = range(1, 61)
        ac = [s.autocorr(lag=k) for k in lags]
        ax.plot(lags, ac, label=f'{mat}Y')
    ax.axhline(0, color='black', lw=0.8)
    ax.axhline(0.5, color='grey', lw=0.6, ls='--', label='half-life threshold')
    ax.set_xlabel('Lag (days)')
    ax.set_ylabel('Autocorrelation')
    ax.set_title(title)
    ax.legend(fontsize=9)

fig.suptitle('Residual autocorrelation by lag')
plt.tight_layout()
plt.savefig('data/week5_day2_autocorr.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Distribution shape

Key things to look for:
- **Fat tails** — a normal distribution at ±2σ captures 95% of observations;
  fat tails mean more extreme events than that. This matters for threshold
  setting in Day 3.
- **Skewness** — asymmetric tails indicate the model misfits in one direction
  more than the other at that maturity.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 12))
axes = axes.flatten()

for i, mat in enumerate(MATURITIES):
    ax = axes[i]
    s = pivot[mat].dropna()
    ax.hist(s, bins=80, color='steelblue', alpha=0.7, density=True)

    # overlay normal with same mean/std
    x = np.linspace(s.min(), s.max(), 300)
    from scipy.stats import norm
    ax.plot(x, norm.pdf(x, s.mean(), s.std()), 'r-', lw=1.5, label='Normal')
    ax.axvline(0, color='black', lw=0.8, ls='--')

    skew = s.skew()
    kurt = s.kurt()  # excess kurtosis; 0 = normal
    ax.set_title(f'{mat}Y  |  skew={skew:.2f}  kurt={kurt:.1f}')
    ax.set_xlabel('residual (bp)')
    ax.legend(fontsize=8)

fig.suptitle('Residual distributions vs Normal', y=1.01)
plt.tight_layout()
plt.savefig('data/week5_day2_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. On-the-run richness: benchmark vs off-benchmark

The on-the-run richness phenomenon predicts that **benchmark maturities**
(2Y/5Y/10Y/30Y) should trade persistently **rich** (negative residual) because
investors pay a liquidity premium for the most recently issued bonds. Check
this by comparing the pre-demeaning means (stored in `residual_means.parquet`)
and the fraction of time each maturity spends cheap vs rich.

In [ ]:
summary = []
for mat in MATURITIES:
    s = pivot[mat].dropna()
    summary.append({
        'maturity':       mat,
        'is_benchmark':   mat in BENCHMARKS,
        'structural_mean_bps': means[mat],   # pre-demeaning bias
        'pct_cheap':      (s > 0).mean(),
        'pct_rich':       (s < 0).mean(),
        'std_bps':        s.std(),
    })

summary_df = pd.DataFrame(summary).set_index('maturity')
print(summary_df.round(3))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Structural mean (pre-demeaning)
colors = ['firebrick' if m in BENCHMARKS else 'steelblue' for m in MATURITIES]
ax1.bar(MATURITIES, means[MATURITIES], color=colors, width=0.6)
ax1.axhline(0, color='black', lw=0.8)
ax1.set_xticks(MATURITIES)
ax1.set_xlabel('Maturity (years)')
ax1.set_ylabel('Mean residual (bp)')
ax1.set_title('Structural bias (pre-demeaning mean)\nred = benchmark maturity')

# pct cheap
pct_cheap = summary_df['pct_cheap']
ax2.bar(MATURITIES, pct_cheap, color=colors, width=0.6)
ax2.axhline(0.5, color='black', lw=0.8, ls='--', label='50% line')
ax2.set_xticks(MATURITIES)
ax2.set_xlabel('Maturity (years)')
ax2.set_ylabel('Fraction of days cheap (residual > 0)')
ax2.set_title('Pct cheap after demeaning\nred = benchmark maturity')
ax2.set_ylim(0, 1)
ax2.legend()

plt.tight_layout()
plt.savefig('data/week5_day2_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Rolling standard deviation — regime changes

If the std of residuals is unstable over time, a fixed ±2σ threshold will
fire too often in calm periods and too rarely in volatile ones. This is why
Day 3 uses a **rolling** 60-day std rather than a full-history std.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
axes = axes.flatten()

for ax, mat in zip(axes, BENCHMARKS):
    rolling_std = pivot[mat].dropna().rolling(60).std()
    ax.plot(rolling_std.index, rolling_std.values, lw=0.8, color='steelblue')
    ax.set_title(f'{mat}Y — 60-day rolling std (bp)')
    ax.set_ylabel('std (bp)')

fig.suptitle('Rolling volatility of residuals at benchmark maturities')
plt.tight_layout()
plt.savefig('data/week5_day2_rolling_std.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Observations

**Persistence:** Lag-1 autocorrelation is 0.92–0.98 at every maturity,
implying half-lives of 9–14 days. Deviations persist long enough to trade
on a weekly rebalancing cycle.

**On-the-run richness:** The clearest signal of on-the-run richness is at 10Y
(structural mean ~+0.19bp raw, and only 18% of days are cheap after demeaning —
meaning the distribution is clustered below zero). The 30Y showed the largest
structural richness bias (−8bp pre-demeaning). The 2Y and 5Y run *cheap*
relative to our Svensson curve most of the time — this is likely a fitting
artifact: our optimizer sacrifices accuracy at the short-medium end to improve
the long-end fit, so the 2Y/5Y observed rates systematically exceed our curve.

**Fat tails:** Excess kurtosis is positive at all maturities, meaning more
extreme moves than a normal distribution predicts. This argues for tighter
entry thresholds (±2σ is fine) and the 30bp outlier cap we already apply.

**Regime changes:** Rolling std at benchmark maturities is visibly higher
around 2008–2009 and 2020–2022. The 60-day rolling window in Day 3 adapts
to these regimes automatically, so no extra adjustment is needed.